# ДЗ 1. Мой первый агент: учимся работать с инструментами

В этом задании вы соберёте агента-планировщика поездок: он подсказывает, куда поехать, ищет маршруты, пересчитывает цены в местную валюту и ведёт маршрут поездки. По дороге попробуете все основные способы описать инструмент для LLM — от «сырой» JSON-схемы до класса-наследника `BaseTool` и увидите, что из этого получает модель.

Все данные (города, маршруты, курсы, климат) лежат в локальном файле `hw01_travel_data.json`.

### Что сдать

Только этот ноутбук с выполненными ячейками: все выводы, включая трейсы агента из части 5, должны быть видны.

### Правила

* Код пишите в местах с пометкой `# TODO`. **Не меняйте имена и сигнатуры** функций и классов: по ним работают автопроверки.
* После большинства заданий стоит ячейка `check(...)`. Она проверяет код без обращения к LLM и печатает ✅ или ❌ с причиной. ✅ нужен для полного балла, но его мало: проверяющий смотрит ещё описания, трейсы и выводы.
* Тексты пишите своими словами и по своим экспериментам.
* Библиотеки: только `langchain` / `langchain-openai` (и то, что они подтягивают).

## 0. Подготовка

In [1]:
%pip install -q "langchain>=1.4,<2" "langchain-openai>=1.7,<2"

In [2]:
from pathlib import Path

# Эти файлы должны лежать рядом с ноутбуком
REQUIRED_FILES = ["hw01_checks.py", "hw01_travel_data.json"]
missing = [name for name in REQUIRED_FILES if not Path(name).exists()]
if missing:
    try:
        from google.colab import files

        print("Загрузите файлы:", ", ".join(missing))
        files.upload()
    except ImportError as error:
        raise FileNotFoundError(f"Положите рядом с ноутбуком: {', '.join(missing)}") from error

In [3]:
import json
import os
from getpass import getpass
from typing import Annotated, Any, Literal, Optional

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import BaseTool, ToolException, tool
from langchain_core.tools import StructuredTool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_openai import ChatOpenAI
from langgraph.errors import GraphRecursionError
from pydantic import BaseModel, Field, PrivateAttr

from hw01_checks import ScriptedChatModel, check, tool_call

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
    os.environ["LLM_MODEL"] = "openai/gpt-oss-120b"
except Exception as e:
    print(e)

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-oss:120b-cloud")
API_KEY = os.environ.setdefault("LLM_API_KEY", "ollama")

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=os.environ["LLM_API_KEY"],
    temperature=0,
)

print(llm.invoke("Ответь одним словом: готово").content)

Готово


### Данные

Ниже — готовые вспомогательные функции. Их не нужно менять, используйте их в своих инструментах:

* `TRAVEL_DATA` — весь датасет; `CITIES`, `ROUTES`, `RATES` — его части;
* `normalize_city(name)` — приводит «питер», «Istanbul», «ЕРЕВАН» к ключу из `CITIES` или возвращает `None`;
* `MONTHS_IN` — названия месяцев в предложном падеже («в январе»).

In [4]:
TRAVEL_DATA = json.loads(Path("hw01_travel_data.json").read_text(encoding="utf-8"))
CITIES: dict = TRAVEL_DATA["cities"]
ROUTES: list = TRAVEL_DATA["routes"]
RATES: dict = TRAVEL_DATA["rates_per_eur"]

TRANSPORT_RU = {"plane": "самолёт", "train": "поезд", "bus": "автобус"}
MONTHS_IN = [
    "январе", "феврале", "марте", "апреле", "мае", "июне",
    "июле", "августе", "сентябре", "октябре", "ноябре", "декабре",
]


def normalize_city(name: str) -> Optional[str]:
    """Возвращает название города, как оно записано в CITIES, или None."""
    key = name.strip().lower().replace("ё", "е")
    for city, info in CITIES.items():
        variants = [city, *info["aliases"]]
        if any(key == variant.lower().replace("ё", "е") for variant in variants):
            return city
    return None


print(TRAVEL_DATA["_about"])
print("Города:", ", ".join(CITIES))
print("Валюты:", RATES)
print("Пример маршрута:", ROUTES[0])
print("Пример города:", json.dumps(CITIES["Тбилиси"], ensure_ascii=False)[:300], "...")

Учебный датасет для ДЗ 1. Цены, курсы и климат вымышлены для учебных целей. Маршруты двусторонние, цены маршрутов в EUR. Курсы: сколько единиц валюты за 1 EUR.
Города: Москва, Санкт-Петербург, Казань, Стамбул, Тбилиси, Ереван, Белград, Дубай
Валюты: {'EUR': 1, 'USD': 1.08, 'RUB': 100, 'TRY': 45, 'GEL': 3, 'AMD': 420, 'RSD': 117, 'AED': 4}
Пример маршрута: {'origin': 'Москва', 'destination': 'Санкт-Петербург', 'transport': 'train', 'duration_hours': 4, 'price_eur': 40}
Пример города: {"aliases": ["Tbilisi"], "country": "Грузия", "currency": "GEL", "utc_offset": 4, "hotel_per_night": 150, "description": "Старый город, серные бани и грузинская кухня.", "attractions": [{"name": "Крепость Нарикала", "price": 0}, {"name": "Серные бани", "price": 80}, {"name": "Канатная дорога", "pric ...


## 1. Агентный цикл с нуля (2 балла)

Прежде чем пользоваться готовым `create_agent`, соберём агента руками, чтобы увидеть протокол tool calling целиком:

1. Модели передают список инструментов (их JSON-схемы) — `llm.bind_tools(tools)`.
2. Модель отвечает `AIMessage`. Если ей нужен инструмент, в `response.tool_calls` будет список вида `{"name": ..., "args": {...}, "id": ...}`. Сама модель ничего не выполняет.
3. Мы вызываем функцию и кладём результат в историю как `ToolMessage(content=..., tool_call_id=<тот же id>)`.
4. Снова вызываем модель с обновлённой историей. Повторяем, пока модель не ответит текстом без `tool_calls`.

Это и есть ReAct-цикл: рассуждение → действие → наблюдение → …

Чтобы агенту было что вызывать, нужен хотя бы один инструмент. В этой части он будет один — пересчёт валют `convert_currency`. Это обычная Python-функция: про агентские фреймворки (к примеру, LangChain) она ничего не знает. Дальше вы опишете её для модели (1.1) и напишете цикл, в котором модель сможет её вызывать (1.2).

In [5]:
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Конвертирует сумму между валютами по курсам из учебного датасета."""
    source, target = from_currency.strip().upper(), to_currency.strip().upper()
    unknown = [code for code in (source, target) if code not in RATES]
    if unknown:
        raise ToolException(f"Неизвестная валюта: {', '.join(unknown)}. Доступны: {', '.join(RATES)}")
    result = amount / RATES[source] * RATES[target]
    return f"{amount:.2f} {source} = {result:.2f} {target}"


convert_currency(100, "EUR", "RUB")

'100.00 EUR = 10000.00 RUB'

### 1.1. JSON-схема инструмента (0.5 балла)

Первый и самый низкоуровневый способ передать инструмент модели — описать его JSON-схемой в формате OpenAI function calling. Модель не видит Python-код: она получает только имя инструмента, описание и схему аргументов. Какой бы способ задать инструмент вы ни выбрали дальше (декоратор, Pydantic, класс), LangChain в итоге превратит его именно в такую схему.

Опишите `convert_currency` в этом формате вручную, словарём:

```python
{"type": "function", "function": {"name": ..., "description": ..., "parameters": <JSON Schema>}}
```

Аргументы описываются внутри `parameters` в формате JSON Schema: `parameters` — это объект (`"type": "object"`), у которого в `properties` перечислены аргументы, а в `required` — обязательные из них. Например, для функции `get_weather(city: str, days: int)` схема выглядит так:

```python
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Возвращает прогноз погоды в городе на несколько дней вперёд.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Название города, например «Казань»"},
                "days": {"type": "integer", "description": "На сколько дней прогноз, от 1 до 7"},
            },
            "required": ["city", "days"],
        },
    },
}
```

Для `convert_currency`:

* в `properties` три аргумента: `amount` (number), `from_currency` и `to_currency` (string), все три обязательные;
* у инструмента и у каждого аргумента должно быть описание. Подумайте, что модели нужно знать, чтобы правильно заполнить `from_currency`: «рубли» или `RUB`?

In [6]:
CONVERT_CURRENCY_SCHEMA = {
    # TODO 1.1: опишите convert_currency в формате OpenAI function calling
    "type": "function",
    "function": {
        "name": "convert_currency",
        "description": "Переводит сумму из одной валюты в другую по курсам учебного датасета.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "Сумма для конвертации"},
                "from_currency": {"type": "string", "description": "Код исходной валюты, например EUR или RUB"},
                "to_currency": {"type": "string", "description": "Код целевой валюты, например GEL или AMD"}
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    }
}

check("1.1", schema=CONVERT_CURRENCY_SCHEMA)

✅ Задание 1.1: базовые тесты пройдены


### 1.2. ReAct-цикл (1.5 балла)

Напишите `run_manual_agent(llm, question, tools, registry, max_steps=5, max_tool_errors=2)`.

* **Вход:** модель, вопрос пользователя, список схем инструментов (`tools`), словарь `registry` «имя инструмента → Python-функция», лимит обращений к модели и лимит ошибок инструментов подряд.
* **Выход:** `dict` с ключами `answer` (текст финального ответа), `messages` (вся история, включая `ToolMessage`), `steps` (сколько раз вызвали модель).
* История начинается с `SystemMessage(MANUAL_SYSTEM_PROMPT)` и `HumanMessage(question)`.
* На каждый вызов из `tool_calls` в историю добавляется ровно один `ToolMessage` с тем же `tool_call_id`.

**Ошибки.** Модель может ошибиться в аргументах (например, передать «рубли» вместо `RUB`) и исправиться, если увидит текст ошибки. Но она может и повторять один и тот же неудачный вызов, сжигая шаги. Поэтому:

* если инструмент бросил исключение, текст ошибки уходит модели в `ToolMessage`;
* если модель вызвала несуществующий инструмент, в `ToolMessage` уходит сообщение «такого инструмента нет» со списком доступных — это тоже ошибка;
* считайте ошибки подряд: каждый неудачный вызов увеличивает счётчик, успешный — обнуляет;
* когда после обработки ответа модели счётчик достиг `max_tool_errors`, цикл останавливается, а в `answer` — понятное пользователю сообщение с текстом последней ошибки.

**Лимит шагов.** Если за `max_steps` обращений модель так и не ответила текстом, верните в `answer` понятное сообщение (не `None`).

Для отладки без трат на API в `hw01_checks` есть `ScriptedChatModel` — фейковая модель, которая отвечает заранее заданными сообщениями (пример в docstring).


<details>
<summary><strong>Подсказка</strong></summary>

`for step in range(1, max_steps + 1): response = model.invoke(messages) ...` — и не забудьте добавить сам `response` в историю перед `ToolMessage`: модель должна видеть, какой вызов она сделала. Если в одном ответе несколько `tool_calls`, обработайте их все и только потом проверяйте счётчик ошибок.

</details>

In [7]:
MANUAL_SYSTEM_PROMPT = (
    "Ты помощник по путешествиям. Для пересчёта валют обязательно используй инструмент, "
    "не считай курсы сам. Отвечай кратко, по-русски."
)


def run_manual_agent(
    llm, question: str, tools: list, registry: dict, max_steps: int = 5, max_tool_errors: int = 2
) -> dict:
    """Минимальный ReAct-агент без фреймворка агентов.

    Args:
        llm: Chat-модель LangChain с поддержкой bind_tools.
        question: Вопрос пользователя.
        tools: Схемы инструментов для модели.
        registry: Имя инструмента → Python-функция.
        max_steps: Максимальное число обращений к модели.
        max_tool_errors: После скольких ошибок инструментов подряд остановиться.

    Returns:
        {"answer": str, "messages": list, "steps": int}
    """
    # TODO 1.2: привяжите инструменты к модели и соберите начальную историю
    model = llm.bind_tools(tools)
    messages = [SystemMessage(content=MANUAL_SYSTEM_PROMPT), HumanMessage(content=question)]
    errors = 0
    last_error = ""

    # TODO 1.2: цикл «модель → инструменты → модель», не больше max_steps обращений к модели;
    #           считайте ошибки инструментов подряд и остановитесь, когда их станет max_tool_errors
    for step in range(1, max_steps + 1):
        response = model.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            if response.text:
                return {"answer": response.text, "messages": messages, "steps": step}
            continue
        for call in response.tool_calls:
            try:
                if call["name"] not in registry:
                    raise ToolException(f"Инструмента {call['name']} нет. Доступны: {', '.join(registry)}")
                result = registry[call["name"]](**call["args"])
                errors = 0
                content = str(result)
                status = "success"
            except Exception as error:
                last_error = str(error)
                errors += 1
                content = last_error
                status = "error"
            messages.append(ToolMessage(content=content, tool_call_id=call["id"], name=call["name"], status=status))
        if errors >= max_tool_errors:
            return {"answer": f"Не удалось выполнить запрос: {last_error}", "messages": messages, "steps": step}
    return {"answer": "Не удалось завершить запрос за отведённое число шагов.", "messages": messages, "steps": max_steps}


check(
    "1.2",
    run_manual_agent=run_manual_agent,
    schema=CONVERT_CURRENCY_SCHEMA,
    registry={"convert_currency": convert_currency},
)

✅ Задание 1.2: базовые тесты пройдены


Запустим на настоящей модели. Вопрос требует двух вызовов инструмента: посмотрите, делает ли модель их параллельно (два `tool_calls` в одном ответе) или по очереди.

In [8]:
result = run_manual_agent(
    llm,
    "Сколько будет 250 евро в грузинских лари и в армянских драмах?",
    tools=[CONVERT_CURRENCY_SCHEMA],
    registry={"convert_currency": convert_currency},
)
for message in result["messages"]:
    message.pretty_print()
print("Обращений к модели:", result["steps"])

================================ System Message ================================

Ты помощник по путешествиям. Для пересчёта валют обязательно используй инструмент, не считай курсы сам. Отвечай кратко, по-русски.
================================ Human Message =================================

Сколько будет 250 евро в грузинских лари и в армянских драмах?
================================== Ai Message ==================================
Tool Calls:
  convert_currency (fc_bee0769e-15ec-4b0a-9b7e-17eee1b0a101)
 Call ID: fc_bee0769e-15ec-4b0a-9b7e-17eee1b0a101
  Args:
    amount: 250
    from_currency: EUR
    to_currency: GEL
================================= Tool Message =================================
Name: convert_currency

250.00 EUR = 750.00 GEL
================================== Ai Message ==================================
Tool Calls:
  convert_currency (fc_c14cb722-be1a-4cbc-a3a9-6f5256a4fe83)
 Call ID: fc_c14cb722-be1a-4cbc-a3a9-6f5256a4fe83
  Args:
    amount: 250
    from_curr

## 2. Пять способов задать инструмент (3 балла)

Писать JSON-схему руками утомительно и легко ошибиться. LangChain умеет строить её сам. Попробуем все основные способы на инструментах планировщика:

| № | Способ | Инструмент |
|---|---|---|
| 2.1 | `@tool` + аннотации типов + docstring | `get_city_info` |
| 2.2 | `@tool(..., args_schema=PydanticModel)` | `search_routes` |
| 2.3 | `StructuredTool.from_function(...)` | `convert_currency_tool` |
| 2.4 | Класс-наследник `BaseTool` с состоянием | `ItineraryTool` |
| 2.5 | Починка плохо описанного инструмента | `get_climate` |

Для каждого инструмента полезно смотреть, что именно увидит модель: `convert_to_openai_tool(tool)` возвращает ту же структуру, что вы писали руками в 1.1.

Во всех инструментах неизвестный город — это ошибка: бросайте `ToolException` с понятным текстом (что не так и какие города есть). Что с ней делать дальше, разберёмся в части 4.

### 2.1. Декоратор `@tool` (0.5 балла)

Напишите `get_city_info(city: str) -> str`: справку о городе — страну, валюту, часовой пояс, цену отеля за ночь, описание и достопримечательности с ценами (0 — «бесплатно»).

* Имя и описание инструмента берутся из имени функции и docstring.
* Чтобы у аргумента тоже появилось описание, используйте `@tool(parse_docstring=True)` и раздел `Args:` в docstring (Google style).
* Город ищите через `normalize_city`.

In [9]:
# TODO 2.1: декоратор и docstring с разделом Args
@tool(parse_docstring=True)
def get_city_info(city: str) -> str:
    """Возвращает справку о городе, отеле и достопримечательностях.

    Args:
        city: Название города или его другое известное название.
    """
    # TODO 2.1: найдите город через normalize_city; если его нет — ToolException
    name = normalize_city(city)
    if name is None:
        raise ToolException(f"Город {city} не найден. Доступны: {', '.join(CITIES)}")
    info = CITIES[name]
    # TODO 2.1: строка вида «Каскад (бесплатно); Матенадаран (2500 AMD); ...»
    attractions = "; ".join(f"{s['name']} ({str(s['price']) + ' ' + info['currency'] if s['price'] else 'бесплатно'})" for s in info["attractions"])
    return (
        f"{name}, {info['country']}. Валюта: {info['currency']}. Часовой пояс: UTC+{info['utc_offset']}.\n"
        f"Отель: около {info['hotel_per_night']} {info['currency']} за ночь.\n"
        f"{info['description']}\n"
        f"Достопримечательности: {attractions}"
    )


check("2.1", get_city_info=get_city_info)
print(get_city_info.invoke({"city": "ереван"}))
print(convert_to_openai_tool(get_city_info))

✅ Задание 2.1: базовые тесты пройдены
Ереван, Армения. Валюта: AMD. Часовой пояс: UTC+4.
Отель: около 25000 AMD за ночь.
Розовый город с видом на Арарат.
Достопримечательности: Каскад (бесплатно); Матенадаран (2500 AMD); Храм Гарни (1500 AMD)
{'type': 'function', 'function': {'name': 'get_city_info', 'description': 'Возвращает справку о городе, отеле и достопримечательностях.', 'parameters': {'properties': {'city': {'description': 'Название города или его другое известное название.', 'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


### 2.2. Явная схема через Pydantic (0.5 балла)

Когда у аргументов есть ограничения (допустимые значения, диапазоны, значения по умолчанию), удобнее описать их Pydantic-моделью и передать через `args_schema`.

**Pydantic-модель данных** — это класс-наследник `pydantic.BaseModel`, поля которого описывают данные: тип, значение по умолчанию, ограничения и описание (`Field(...)`). LangChain строит из неё JSON-схему аргументов для модели, а перед вызовом инструмента проверяет, что аргументы, которые сгенерировала модель, ей соответствуют. Если нет — инструмент не вызывается, а модель получает текст ошибки валидации.

Напишите модель данных `RouteQuery` и инструмент `search_routes`:

* `origin`, `destination` — обязательные строки;
* `transport` — `Literal["plane", "train", "bus", "any"]`, по умолчанию `"any"`;
* `max_price_eur` — необязательное число, `≥ 0` (`Field(ge=0)`), по умолчанию `None`;
* у всех полей есть `description`.

Инструмент ищет прямые маршруты, учитывая, что маршруты двусторонние (Москва → Казань = Казань → Москва), фильтрует по транспорту и цене и возвращает их отсортированными по цене, по строке на маршрут, например `Москва → Стамбул: самолёт, 4 ч, 250 EUR`. Если ничего не нашлось — понятный текст «маршрутов не найдено».

Подумайте, что написать в описании инструмента, чтобы агент догадался искать пересадку, когда прямого маршрута нет.

In [10]:
class RouteQuery(BaseModel):
    """Параметры поиска прямых маршрутов между двумя городами."""

    # TODO 2.2: поля origin, destination, transport, max_price_eur с описаниями и ограничениями
    origin: str = Field(description="Город отправления")
    destination: str = Field(description="Город назначения")
    transport: Literal["plane", "train", "bus", "any"] = Field(default="any", description="Вид транспорта или любой")
    max_price_eur: Optional[float] = Field(default=None, ge=0, description="Максимальная цена в евро")


# TODO 2.2: декоратор с args_schema, содержательный docstring и реализация поиска маршрутов
@tool(args_schema=RouteQuery)
def search_routes(
    origin: str, destination: str, transport: str = "any", max_price_eur: Optional[float] = None
) -> str:
    """Ищет прямые маршруты между городами в обе стороны; если прямого нет, попробуйте пересадку через другой город."""
    start, end = normalize_city(origin), normalize_city(destination)
    if start is None or end is None:
        raise ToolException(f"Неизвестный город. Доступны: {', '.join(CITIES)}")
    found = [route for route in ROUTES if {route["origin"], route["destination"]} == {start, end} and (transport == "any" or route["transport"] == transport) and (max_price_eur is None or route["price_eur"] <= max_price_eur)]
    if not found:
        return "Прямых маршрутов не найдено. Попробуйте пересадку через другой город."
    return "\n".join(f"{start} → {end}: {TRANSPORT_RU[route['transport']]}, {route['duration_hours']} ч, {route['price_eur']} EUR" for route in sorted(found, key=lambda route: route["price_eur"]))


check("2.2", search_routes=search_routes, RouteQuery=RouteQuery)
print(search_routes.invoke({"origin": "Питер", "destination": "Москва"}))
print(convert_to_openai_tool(search_routes))

✅ Задание 2.2: базовые тесты пройдены
Санкт-Петербург → Москва: автобус, 10 ч, 20 EUR
Санкт-Петербург → Москва: поезд, 4 ч, 40 EUR
Санкт-Петербург → Москва: самолёт, 1.5 ч, 60 EUR
{'type': 'function', 'function': {'name': 'search_routes', 'description': 'Ищет прямые маршруты между городами в обе стороны; если прямого нет, попробуйте пересадку через другой город.', 'parameters': {'properties': {'origin': {'description': 'Город отправления', 'type': 'string'}, 'destination': {'description': 'Город назначения', 'type': 'string'}, 'transport': {'default': 'any', 'description': 'Вид транспорта или любой', 'enum': ['plane', 'train', 'bus', 'any'], 'type': 'string'}, 'max_price_eur': {'anyOf': [{'minimum': 0, 'type': 'number'}, {'type': 'null'}], 'default': None, 'description': 'Максимальная цена в евро'}}, 'required': ['origin', 'destination'], 'type': 'object'}}}


### 2.3. `StructuredTool.from_function` (0.5 балла)

Бывает, что функция уже есть (из чужого модуля, из части 1) и менять её код нельзя. Тогда инструмент собирают снаружи. Оберните `convert_currency` из части 1 в `StructuredTool.from_function`, задав имя `convert_currency` и содержательное `description`.

Сравните получившуюся схему со своей схемой из 1.1: чего в ней не хватает и почему? Ответ запишите в 2.6.

In [11]:
# TODO 2.3: оберните convert_currency, не меняя её код
convert_currency_tool = StructuredTool.from_function(
    convert_currency,
    name="convert_currency",
    description="Конвертирует суммы между валютами. from_currency и to_currency передавайте трёхбуквенными ISO-кодами валют, например EUR, RUB, GEL."
)

check("2.3", convert_currency_tool=convert_currency_tool, convert_currency=convert_currency)
print(convert_to_openai_tool(convert_currency_tool))

✅ Задание 2.3: базовые тесты пройдены
{'type': 'function', 'function': {'name': 'convert_currency', 'description': 'Конвертирует суммы между валютами. from_currency и to_currency передавайте трёхбуквенными ISO-кодами валют, например EUR, RUB, GEL.', 'parameters': {'properties': {'amount': {'type': 'number'}, 'from_currency': {'type': 'string'}, 'to_currency': {'type': 'string'}}, 'required': ['amount', 'from_currency', 'to_currency'], 'type': 'object'}}}


### 2.4. Наследник `BaseTool` с состоянием (0.5 балла)

Класс нужен, когда инструменту требуется состояние или конфигурация — подключение к БД, клиент API, накопленные данные. Наш случай: маршрут поездки, который агент собирает за несколько вызовов.

Напишите `ItineraryTool(BaseTool)` с именем `manage_itinerary` и схемой аргументов `ItineraryInput`:

* `action`: `Literal["add", "show", "clear"]`;
* `city`: необязательная строка (нужна для `add`);
* `nights`: необязательное целое от 1 до 30 (нужно для `add`).

Поведение:

* конфигурация — поле `data: dict` (весь `TRAVEL_DATA`), передаётся в конструктор: `ItineraryTool(data=TRAVEL_DATA)`;
* состояние — список остановок в `PrivateAttr`: у каждого экземпляра свой маршрут;
* `add` добавляет город и число ночей; неизвестный город или отсутствие `nights` → `ToolException`;
* `clear` очищает маршрут;
* `show` возвращает маршрут в порядке добавления и итог. Стоимость отеля: `ночи × hotel_per_night / курс валюты города` (курсы — единиц за 1 EUR). Последние две строки строго в формате:

```
Маршрут поездки:
1. Тбилиси — 3 ноч., отель ≈ 150.00 EUR
2. Ереван — 2 ноч., отель ≈ 119.05 EUR
Итого ночей: 5
Стоимость отелей: ≈ 269.05 EUR
```

Для пустого маршрута `show` выводит `Маршрут пуст.` и те же две итоговые строки: `Итого ночей: 0` и `Стоимость отелей: ≈ 0.00 EUR`.


<details>
<summary><strong>Подсказка</strong></summary>

В `BaseTool` поля объявляются как в Pydantic-модели: `name: str = "manage_itinerary"`, `description: str = ...`, `args_schema: type[BaseModel] = ItineraryInput`. Логика — в методе `_run(self, action, city=None, nights=None) -> str`. Состояние храните в `PrivateAttr(default_factory=list)`: обычное поле стало бы частью конфигурации инструмента (его можно передать в конструктор, оно попадает в `repr` и сериализацию), а это внутренние данные.

</details>

In [12]:
class ItineraryInput(BaseModel):
    # TODO 2.4: поля action, city, nights с описаниями и ограничениями
    action: Literal["add", "show", "clear"] = Field(description="Добавить остановку, показать или очистить маршрут")
    city: Optional[str] = Field(default=None, description="Город остановки для добавления")
    nights: Optional[int] = Field(default=None, ge=1, le=30, description="Число ночей от 1 до 30")


class ItineraryTool(BaseTool):
    # TODO 2.4: name, description, args_schema, поле data и состояние в PrivateAttr
    name: str = "manage_itinerary"
    description: str = "Добавляет города и ночи в маршрут поездки, показывает маршрут и стоимость отелей в евро или очищает маршрут."
    args_schema: type[BaseModel] = ItineraryInput
    data: dict
    _stops: list = PrivateAttr(default_factory=list)

    def _run(self, action: str, city: Optional[str] = None, nights: Optional[int] = None, run_manager=None) -> str:
        # TODO 2.4: add / show / clear
        if action == "clear":
            self._stops.clear()
            return "Маршрут очищен."
        if action == "add":
            name = normalize_city(city or "")
            if name is None:
                raise ToolException(f"Город {city} не найден")
            if nights is None:
                raise ToolException("Укажите число ночей")
            self._stops.append((name, nights))
            return f"Добавлено: {name} — {nights} ноч."
        lines = ["Маршрут поездки:"] if self._stops else ["Маршрут пуст."]
        total_nights = 0
        total_eur = 0
        for i, (name, count) in enumerate(self._stops, 1):
            info = self.data["cities"][name]
            cost = count * info["hotel_per_night"] / self.data["rates_per_eur"][info["currency"]]
            lines.append(f"{i}. {name} — {count} ноч., отель ≈ {cost:.2f} EUR")
            total_nights += count
            total_eur += cost
        lines.append(f"Итого ночей: {total_nights}")
        lines.append(f"Стоимость отелей: ≈ {total_eur:.2f} EUR")
        return "\n".join(lines)


check("2.4", ItineraryTool=ItineraryTool)
itinerary_tool = ItineraryTool(data=TRAVEL_DATA)
print(convert_to_openai_tool(itinerary_tool))

✅ Задание 2.4: базовые тесты пройдены
{'type': 'function', 'function': {'name': 'manage_itinerary', 'description': 'Добавляет города и ночи в маршрут поездки, показывает маршрут и стоимость отелей в евро или очищает маршрут.', 'parameters': {'properties': {'action': {'description': 'Добавить остановку, показать или очистить маршрут', 'enum': ['add', 'show', 'clear'], 'type': 'string'}, 'city': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Город остановки для добавления'}, 'nights': {'anyOf': [{'maximum': 30, 'minimum': 1, 'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Число ночей от 1 до 30'}}, 'required': ['action'], 'type': 'object'}}}


### 2.5. Починить плохой инструмент (0.5 балла)

Вот инструмент, который «работает». Посмотрите на его схему: что из неё может понять модель?

In [13]:
@tool
def climate(c, m):
    """climate"""
    return CITIES[c]["climate"][m - 1]


print(climate.invoke({"c": "Тбилиси", "m": 5}))
print(convert_to_openai_tool(climate))

17
{'type': 'function', 'function': {'name': 'climate', 'description': 'climate', 'parameters': {'properties': {'c': {}, 'm': {}}, 'required': ['c', 'm'], 'type': 'object'}}}


Перепишите его как `get_climate(city, month) -> str`:

* понятное имя, типы аргументов, описание инструмента и каждого аргумента;
* `month` — целое от 1 до 12 прямо в схеме (подсказка: `Annotated[int, Field(ge=1, le=12)]`);
* город — через `normalize_city`, неизвестный город → `ToolException`;
* ответ — фраза вида «Средняя температура в городе Дубай в июле: +35 °C» (`MONTHS_IN` уже есть).

Здесь можно использовать любой из способов выше — выберите тот, что кажется уместнее, и объясните выбор в 2.6.

In [14]:
# TODO 2.5: перепишите climate как get_climate
@tool(parse_docstring=True)
def get_climate(city: str, month: Annotated[int, Field(ge=1, le=12)]) -> str:
    """Возвращает среднюю температуру в городе за указанный месяц, не прогноз погоды.

    Args:
        city: Название города.
        month: Номер месяца от 1 до 12.
    """
    name = normalize_city(city)
    if name is None:
        raise ToolException(f"Город {city} не найден")
    return f"В {name} в {MONTHS_IN[month - 1]} средняя температура {CITIES[name]['climate'][month - 1]} °C"


check("2.5", get_climate=get_climate)
print(get_climate.invoke({"city": "Дубай", "month": 7}))
print(convert_to_openai_tool(get_climate))

✅ Задание 2.5: базовые тесты пройдены
В Дубай в июле средняя температура 35 °C
{'type': 'function', 'function': {'name': 'get_climate', 'description': 'Возвращает среднюю температуру в городе за указанный месяц, не прогноз погоды.', 'parameters': {'properties': {'city': {'description': 'Название города.', 'type': 'string'}, 'month': {'description': 'Номер месяца от 1 до 12.', 'maximum': 12, 'minimum': 1, 'type': 'integer'}}, 'required': ['city', 'month'], 'type': 'object'}}}


### Итоговый набор инструментов

Соберём все инструменты в один список и посмотрим их схемы рядом.

In [15]:
ALL_TOOLS = [get_city_info, search_routes, convert_currency_tool, itinerary_tool, get_climate]

for current_tool in ALL_TOOLS:
    print(json.dumps(convert_to_openai_tool(current_tool), ensure_ascii=False, indent=1))
    print()

{
 "type": "function",
 "function": {
  "name": "get_city_info",
  "description": "Возвращает справку о городе, отеле и достопримечательностях.",
  "parameters": {
   "properties": {
    "city": {
     "description": "Название города или его другое известное название.",
     "type": "string"
    }
   },
   "required": [
    "city"
   ],
   "type": "object"
  }
 }
}

{
 "type": "function",
 "function": {
  "name": "search_routes",
  "description": "Ищет прямые маршруты между городами в обе стороны; если прямого нет, попробуйте пересадку через другой город.",
  "parameters": {
   "properties": {
    "origin": {
     "description": "Город отправления",
     "type": "string"
    },
    "destination": {
     "description": "Город назначения",
     "type": "string"
    },
    "transport": {
     "default": "any",
     "description": "Вид транспорта или любой",
     "enum": [
      "plane",
      "train",
      "bus",
      "any"
     ],
     "type": "string"
    },
    "max_price_eur": {
   

### 2.6. Сравнение способов (0.5 балла)

Ответьте своими словами, опираясь на схемы выше (5–10 предложений):

1. Чем отличаются схемы, полученные разными способами? Чего не хватает схеме `convert_currency_tool` по сравнению с вашей ручной из 1.1 и как это исправить, не трогая функцию?
2. Когда вы выберете `@tool`, когда `args_schema`, когда `StructuredTool.from_function`, а когда наследника `BaseTool`? Приведите по примеру из реальной задачи.
3. Что модель «знает» об инструменте, кроме его схемы? Почему описания так важны?

**Ваш ответ:**

1. В convert_currency_tool, созданном через StructuredTool.from_function, отсутствуют описания отдельных параметров, которые были в ручной схеме 1.1. Исправить это можно, передав Pydantic-модель с описаниями полей через args_schema.

2. @tool удобен для простых функций, например, проверить статус задачи. args_schema лучше использовать в случае большого количества параметров, у которых есть ограничения и которые надо контролировать (например, фильтрация товаров при поиске). StructuredTool.from_function подходит для использования чужой готовой функции (например, функции расчета налога). Наследника BaseTool стоит выбирать, если инструмент должен хранить состояние между вызовами, например для работы с корзиной интернет-магазина.

3. Модель видит название инструмента, его описание, параметры и ограничения, а после вызова получает результат работы. Сам код реализации она не видит, поэтому понятные описания важны для правильного выбора инструмента и передачи аргументов.

## 3. Агент на `create_agent` (2 балла)

Теперь отдадим цикл из части 1 фреймворку. `create_agent(model, tools, system_prompt=...)` из `langchain.agents` собирает ровно тот же ReAct-цикл, но умеет больше: исполняет инструменты параллельно, обрабатывает ошибки валидации аргументов.

**Память диалога.** Агент сам по себе ничего не помнит между вызовами `invoke`: он видит только те сообщения, которые вы ему передали. Поэтому память диалога — это просто список сообщений, как `messages` в части 1. `agent.invoke({"messages": [...]})` возвращает словарь, где `result["messages"]` — вся история: переданные сообщения плюс новые (вызовы инструментов, их результаты, ответ). Чтобы продолжить диалог, передайте эту историю в следующий вызов вместе с новой репликой. Новый диалог — пустой список.

### 3. Системный промпт и агент

1. Напишите `SYSTEM_PROMPT` (не короче 200 символов): роль агента, правила работы с инструментами (факты только из инструментов, что делать, если прямого маршрута нет, в какой валюте цены), формат ответа и что делать с запросами, которые инструментами не решаются.
2. `build_agent(llm, tools)` — возвращает агента из `create_agent` с вашим промптом.
3. `chat(agent, history, message) -> (answer, history)` — добавляет реплику пользователя к истории `history`, вызывает агента и возвращает пару: текст последнего ответа агента и обновлённую историю. Исходный список `history` не изменяйте.

In [16]:
SYSTEM_PROMPT = """
Ты агент-планировщик путешествий. Отвечай по-русски кратко и понятно.
Все факты о городах, климате, маршрутах, ценах и курсах бери только из предоставленных инструментов, ничего не выдумывай.
Для поиска дороги используй search_routes: если прямого маршрута нет, проверяй варианты с пересадкой через другие города, сравнивай суммарную цену.
Цены на транспорт указывай в EUR, другие цены указывай с валютой, при необходимости вызывай convert_currency.
Для планирования ночёвок используй manage_itinerary.
У тебя есть средняя температура по месяцам, но нет прогноза на конкретные даты.
Не обещай покупать билеты или бронировать отели: таких инструментов нет.
Если запрос невозможно выполнить, честно сообщи об ограничении и предложи доступную альтернативу.
Не утверждай, что действие выполнено, без результата инструмента.
"""


def build_agent(llm, tools: list):
    """Собирает ReAct-агента на create_agent."""
    # TODO 3
    return create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)


def chat(agent, history: list, message: str) -> tuple[str, list]:
    """Продолжает диалог: возвращает ответ агента и обновлённую историю."""
    # TODO 3
    result = agent.invoke({"messages": [*history, HumanMessage(content=message)]})
    return result["messages"][-1].text, result["messages"]


check("3", build_agent=build_agent, chat=chat, tools=ALL_TOOLS, SYSTEM_PROMPT=SYSTEM_PROMPT)

✅ Задание 3: базовые тесты пройдены


Проведём диалог из трёх реплик. Каждая следующая опирается на предыдущие — агент должен помнить контекст. В конце убедимся, что с пустой историей агент начинает разговор с чистого листа. Перед этим очищаем и маршрут поездки: состояние инструмента — это не память диалога, и без очистки агент «вспомнит» прошлую поездку через `manage_itinerary`.

In [17]:
itinerary_tool.invoke({"action": "clear"})
agent = build_agent(llm, ALL_TOOLS)

dialog = [
    "Выбираю, куда поехать из Москвы в октябре: Стамбул, Тбилиси или Дубай. Где будет теплее +15?",
    "Давай Стамбул. Сколько стоит туда добраться и сколько это в местной валюте?",
    "Добавь в маршрут 4 ночи там и покажи, сколько выйдут отели.",
]
trip_history = []
for message in dialog:
    print("Пользователь:", message)
    answer, trip_history = chat(agent, trip_history, message)
    print("Агент:", answer)
    print()

# Новый диалог: пустая история и пустой маршрут — иначе агент «вспомнит»
# прошлую поездку через manage_itinerary, а не через память диалога
itinerary_tool.invoke({"action": "clear"})
print("Пользователь (новый диалог): Куда я собирался?")
answer, _ = chat(agent, [], "Куда я собирался?")
print("Агент:", answer)

Пользователь: Выбираю, куда поехать из Москвы в октябре: Стамбул, Тбилиси или Дубай. Где будет теплее +15?
Агент: В октябре средняя температура:

- **Стамбул** – 16 °C  
- **Тбилиси** – 14 °C  
- **Дубай** – 29 °C  

Только в Дубае температура будет выше +15 °C.

Пользователь: Давай Стамбул. Сколько стоит туда добраться и сколько это в местной валюте?
Агент: - **Транспорт:** прямой рейс Москва → Стамбул, самолёт, ≈ 4 ч, **250 EUR**.  
- **В местной валюте (турецкие лиры):** 250 EUR ≈ **11 250 TRY**.  

Пользователь: Добавь в маршрут 4 ночи там и покажи, сколько выйдут отели.
Агент: **Маршрут:** Стамбул — 4 ночи  
**Стоимость отелей:** ≈ 240 EUR (по текущим данным).  

Пользователь (новый диалог): Куда я собирался?
Агент: У меня нет информации о ваших предыдущих планах. Пожалуйста, уточните, какой город, страна или маршрут вас интересует, и я помогу спланировать поездку.


Посмотрите, какие инструменты агент вызывал внутри диалога: вся история лежит в `trip_history`.

In [18]:
for message in trip_history:
    message.pretty_print()

================================ Human Message =================================

Выбираю, куда поехать из Москвы в октябре: Стамбул, Тбилиси или Дубай. Где будет теплее +15?
================================== Ai Message ==================================
Tool Calls:
  get_climate (fc_fd17a434-2b57-4055-999c-061dd5ca324a)
 Call ID: fc_fd17a434-2b57-4055-999c-061dd5ca324a
  Args:
    city: Istanbul
    month: 10
================================= Tool Message =================================
Name: get_climate

В Стамбул в октябре средняя температура 16 °C
================================== Ai Message ==================================
Tool Calls:
  get_climate (fc_d699486a-1190-4164-b651-8f29be7de4d9)
 Call ID: fc_d699486a-1190-4164-b651-8f29be7de4d9
  Args:
    city: Tbilisi
    month: 10
================================= Tool Message =================================
Name: get_climate

В Тбилиси в октябре средняя температура 14 °C
================================== Ai Message ========

## 4. Работа с ошибками (2 балла)

В части 1 вы сами решали, что делать с исключением инструмента. В `create_agent` есть два случая:

* **ошибка валидации аргументов** (например, `max_price_eur=-5`) — агент по умолчанию возвращает модели текст ошибки, и она может исправиться;
* **`ToolException` из кода инструмента** — по умолчанию пробрасывается наружу и роняет весь вызов агента. Проверьте сами ячейкой ниже.

In [19]:
crash_llm = ScriptedChatModel(responses=[tool_call("get_city_info", city="Атлантида"), AIMessage(content="...")])
try:
    build_agent(crash_llm, ALL_TOOLS).invoke({"messages": [{"role": "user", "content": "Расскажи про Атлантиду"}]})
except ToolException as error:
    print("Агент упал:", error)

Агент упал: Город Атлантида не найден. Доступны: Москва, Санкт-Петербург, Казань, Стамбул, Тбилиси, Ереван, Белград, Дубай


### 4.1. Обработка ошибок инструментов (1 балл)

У `BaseTool` есть поле `handle_tool_error`: `True` (вернуть модели текст исключения), строка (вернуть её) или функция `(ToolException) -> str`.

1. Напишите `format_tool_error(error)`: текст для модели, а не для человека — что пошло не так и что ей делать дальше (исправить аргументы, взять другой город, сообщить пользователю). Обязательно включите в него исходный текст исключения (`str(error)`): там сказано, что именно не так, например какой город не найден. Остальная формулировка — на ваше усмотрение.
2. Включите обработку у всех инструментов из `ALL_TOOLS`.
3. Запустите живую проверку ниже и опишите в ячейке после неё, как агент повёл себя с ошибкой.

In [20]:
def format_tool_error(error: ToolException) -> str:
    """Текст ошибки инструмента, который увидит модель."""
    # TODO 4.1
    return f"Ошибка инструмента: {error}. Исправь аргументы, выбери другой город или сообщи пользователю, что запрос не выполнен."


# TODO 4.1: включите обработку ошибок у всех инструментов из ALL_TOOLS
for current_tool in ALL_TOOLS:
    current_tool.handle_tool_error = format_tool_error

check("4.1", tools=ALL_TOOLS, build_agent=build_agent)

✅ Задание 4.1: базовые тесты пройдены


In [21]:
agent = build_agent(llm, ALL_TOOLS)
answer, error_history = chat(agent, [], "Сколько стоит отель в Атлантиде и как туда добраться из Москвы?")
print(answer)
print()
for message in error_history:
    if isinstance(message, ToolMessage):
        print(f"[{message.status}] {message.name}: {message.content[:200]}")

К сожалению, в базе данных нет информации о городе «Атлантида» — его нет в списке доступных городов. Поэтому я не могу предоставить цены на отель и варианты маршрутов из Москвы. Если вы уточните реальный город из доступного списка (Москва, Санкт‑Петербург, Казань, Стамбул, Тбилиси, Ереван, Белград, Дубай), я с радостью помогу подобрать транспорт и отель.

[error] get_city_info: Ошибка инструмента: Город Атлантида не найден. Доступны: Москва, Санкт-Петербург, Казань, Стамбул, Тбилиси, Ереван, Белград, Дубай. Исправь аргументы, выбери другой город или сообщи пользователю, что 


**Что сделал агент после ошибки?** Последовал инструкциям и честно сказал об ошибке и предложил выбрать из существующих городов.

### 4.2. Лимит шагов (0.5 балла)

Агент может зациклиться: например, раз за разом вызывать инструмент с одними и теми же аргументами. У графа LangGraph есть `recursion_limit` (передаётся в `config`), а при его превышении он бросает `GraphRecursionError`.

Напишите `safe_invoke(agent, history, message, max_steps=8) -> (answer, history)` — аналог `chat` с ограничением:

* модель вызывается не больше `max_steps` раз — подберите `recursion_limit` (подсказка: одно обращение к модели и одно исполнение инструментов — это два шага графа);
* в обычном случае возвращает ответ агента и обновлённую историю, как `chat`;
* при превышении лимита функция не падает, а возвращает понятный пользователю текст и исходную историю `history`: незавершённый шаг в неё не попадает.

In [22]:
def safe_invoke(agent, history: list, message: str, max_steps: int = 8) -> tuple[str, list]:
    """Как chat, но не больше max_steps обращений к модели."""
    # TODO 4.2
    try:
        result = agent.invoke({"messages": [*history, HumanMessage(content=message)]}, config={"recursion_limit": 2 * max_steps})
        return result["messages"][-1].text, result["messages"]
    except GraphRecursionError:
        return "Не удалось завершить запрос за отведённое число шагов.", history


check("4.2", safe_invoke=safe_invoke, build_agent=build_agent, tools=ALL_TOOLS)

✅ Задание 4.2: базовые тесты пройдены


### 4.3. Запрос за пределами возможностей (0.5 балла)

Агент должен честно разделять, что он может, а что нет. Самый показательный случай — смешанный запрос: часть агент выполнить может, часть нет, а для ещё одной части у него есть похожие, но не те данные.

Запрос ниже состоит из трёх частей:

1. найти самый дешёвый способ добраться из Москвы в Тбилиси - это агент умеет;
2. сказать, какая погода будет там завтра - у агента есть только средняя температура по месяцам, а не прогноз;
3. купить билет — этого агент не умеет.

Хороший ответ выполняет первую часть, честно говорит, что прогноза погоды у него нет (но может назвать среднюю температуру за месяц), и не делает вид, что купил билет.

Посмотрите на ответ и вызовы инструментов. Если агент где-то выдал желаемое за действительное — доработайте `SYSTEM_PROMPT` в части 3, перезапустите ячейки и опишите, что изменили. Если агент сразу ответил хорошо — опишите, по каким признакам в ответе и вызовах это видно.

In [23]:
agent = build_agent(llm, ALL_TOOLS)
answer, scope_history = safe_invoke(
    agent,
    [],
    "Найди самый дешёвый способ добраться из Москвы в Тбилиси, скажи, какая там будет погода завтра, "
    "и сразу купи мне билет.",
)
for message in scope_history:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            print(f"Вызов: {call['name']}({call['args']})")
print()
print(answer)

Вызов: search_routes({'destination': 'Tbilisi', 'origin': 'Moscow', 'transport': 'any'})
Вызов: search_routes({'destination': 'Tbilisi', 'origin': 'Moscow', 'transport': 'bus'})
Вызов: search_routes({'destination': 'Tbilisi', 'origin': 'Moscow', 'transport': 'train'})

Самый дешёвый найденный прямой вариант — самолёт Москва → Тбилиси, ≈ 220 EUR.

Погода в Тбилиси на 21 ноября (средняя температура за ноябрь): ≈ 10 °C (днём) / ≈ 4 °C (ночью). Точный прогноз на конкретный день у меня нет, только средние значения.

Купить билет я не могу — у меня нет доступа к сервисам бронирования. Вы можете оформить перелёт через любой онлайн‑агент или авиакомпанию, используя указанные данные о цене. Если понадобится подобрать альтернативные маршруты с пересадками, дайте знать.


**Как агент справился с каждой из трёх частей запроса и что пришлось поменять в промпте?** На все три части ответил правильно: правильный ответ на первый вопрос, честно ответил, что нет точной информации, на последний вопрос сказал об отсутствии такого инструмента. Пришлось добавить в промпт инструкцию сообщать об отсутствии запрашиваемых инструментов.

## 5. Прогон на тестовых вопросах (1 балл)

Прогоним агента на фиксированном наборе вопросов и сохраним трейсы — последовательность шагов агента. По трейсам видно, правильно ли агент выбирал инструменты, а не только каков финальный ответ.

### 5. Трейс агента

**Зачем.** После вызова агента вся история разговора лежит в `result["messages"]` — это та же история, которую возвращают `chat` и `safe_invoke`. Это список объектов LangChain: `HumanMessage` (вопрос пользователя), `AIMessage` (ответ модели: либо запрос на вызов инструментов, либо финальный текст), `ToolMessage` (результат инструмента). Читать такой список неудобно, а сохранить его в JSON напрямую нельзя.

Нужна функция, которая превратит историю в трейс — простой список шагов из обычных словарей. По трейсу сразу видно, какие инструменты агент вызывал, с какими аргументами и что получил в ответ. По трейсам в конце части вы разберёте, как агент справился с тестовыми вопросами.

**Что написать.** Функцию `extract_trace(messages) -> list[dict]`:

* вход — список сообщений из истории агента;
* выход — список словарей, по одному на шаг, в том же порядке, что и сообщения.

Каждое сообщение превращается в один или несколько шагов по правилам:

| Сообщение | Шаг(и) в трейсе |
|---|---|
| `SystemMessage` | пропускается |
| `HumanMessage` | `{"type": "user", "content": <текст>}` |
| `AIMessage` **с** `tool_calls` | если у сообщения есть текст — сначала `{"type": "thought", "content": <текст>}`; затем по шагу на каждый вызов: `{"type": "tool_call", "name": <имя>, "args": <аргументы>}` |
| `ToolMessage` | `{"type": "tool_result", "name": <имя инструмента>, "content": <текст>, "is_error": <True, если status == "error">}` |
| `AIMessage` **без** `tool_calls` | `{"type": "answer", "content": <текст>}` |

**Пример.** Пользователь спросил про дорогу в Белград, модель одним ответом запросила два поиска, второй завершился ошибкой, затем модель ответила.

История (вход):

```python
[
    HumanMessage("Сколько стоит дорога из Москвы в Белград?"),
    AIMessage("", tool_calls=[
        {"name": "search_routes", "args": {"origin": "Москва", "destination": "Стамбул"}, "id": "a"},
        {"name": "search_routes", "args": {"origin": "Стамбул", "destination": "Белград"}, "id": "b"},
    ]),
    ToolMessage("plane 250 EUR", tool_call_id="a", name="search_routes"),
    ToolMessage("Ошибка: сервис недоступен", tool_call_id="b", name="search_routes", status="error"),
    AIMessage("Через Стамбул, от 310 EUR."),
]
```

Трейс (выход):

```python
[
    {"type": "user", "content": "Сколько стоит дорога из Москвы в Белград?"},
    {"type": "tool_call", "name": "search_routes", "args": {"origin": "Москва", "destination": "Стамбул"}},
    {"type": "tool_call", "name": "search_routes", "args": {"origin": "Стамбул", "destination": "Белград"}},
    {"type": "tool_result", "name": "search_routes", "content": "plane 250 EUR", "is_error": False},
    {"type": "tool_result", "name": "search_routes", "content": "Ошибка: сервис недоступен", "is_error": True},
    {"type": "answer", "content": "Через Стамбул, от 310 EUR."},
]
```

Здесь у первого `AIMessage` текст пустой, поэтому шага `thought` нет. Именно этот пример проверяет `check("5", ...)`.

<details>
<summary><strong>Подсказка</strong></summary>

Тип сообщения проверяйте через `isinstance(message, HumanMessage)` и т. д. Текст любого сообщения — `message.text`, вызовы — `message.tool_calls` (список словарей с ключами `name`, `args`, `id`), статус результата — `message.status`.

</details>

In [24]:
def extract_trace(messages: list) -> list[dict]:
    """Превращает историю сообщений агента в список шагов."""
    trace = []
    for message in messages:
        # TODO 5: HumanMessage → user; AIMessage с tool_calls → thought (если есть текст) и tool_call на каждый вызов;
        #         ToolMessage → tool_result; AIMessage без tool_calls → answer; SystemMessage пропускаем
        if isinstance(message, HumanMessage):
            trace.append({"type": "user", "content": message.text})
        elif isinstance(message, AIMessage):
            if message.tool_calls:
                if message.text:
                    trace.append({"type": "thought", "content": message.text})
                for call in message.tool_calls:
                    trace.append({"type": "tool_call", "name": call["name"], "args": call["args"]})
            else:
                trace.append({"type": "answer", "content": message.text})
        elif isinstance(message, ToolMessage):
            trace.append({"type": "tool_result", "name": message.name, "content": message.text, "is_error": message.status == "error"})
    return trace


check("5", extract_trace=extract_trace)

✅ Задание 5: базовые тесты пройдены


Тестовые вопросы подобраны так, чтобы проверить разные ситуации. **Не меняйте их.** Каждый вопрос задаётся с пустой историей, а маршрут поездки очищается.

In [25]:
TEST_QUESTIONS = [
    # один инструмент
    "Какая погода в Тбилиси в мае?",
    # два инструмента подряд: валюта города → пересчёт
    "Какая валюта в Ереване и сколько в ней будет 100 евро?",
    # нет прямого маршрута: нужна пересадка
    "Как добраться из Казани в Белград? Нужен самый дешёвый вариант.",
    # состояние: маршрут из нескольких городов
    "Составь маршрут: 3 ночи в Стамбуле и 2 в Белграде. Сколько выйдут отели в евро?",
    # фильтры и честный ответ «не найдено»
    "Сколько стоит поезд из Москвы в Дубай?",
    # ошибка инструмента
    "Расскажи, что посмотреть в Атлантиде.",
    # инструмент не нужен
    "Привет! Что ты умеешь?",
    # вне компетенции
    "Забронируй мне отель в Дубае на завтра.",
]


def run_agent(question: str, max_steps: int = 10) -> dict:
    """Один вопрос с чистого листа: ответ и трейс."""
    itinerary_tool.invoke({"action": "clear"})
    agent = build_agent(llm, ALL_TOOLS)
    answer, messages = safe_invoke(agent, [], question, max_steps=max_steps)
    return {"question": question, "answer": answer, "trace": extract_trace(messages)}


def format_step(step: dict) -> str:
    """Одна строка трейса для печати."""
    kind = step.get("type")
    if kind == "user":
        return f"Пользователь: {step['content']}"
    if kind == "thought":
        return f"Мысль: {step['content'][:200]}"
    if kind == "tool_call":
        args = ", ".join(f"{key}={value!r}" for key, value in step["args"].items())
        return f"Вызов: {step['name']}({args})"
    if kind == "tool_result":
        mark = "Ошибка:" if step.get("is_error") else "Результат:"
        return f"   {mark} {step['content'][:150].replace(chr(10), ' / ')}"
    if kind == "answer":
        return f"Ответ: {step['content'][:400]}"
    return str(step)


results = []
for number, question in enumerate(TEST_QUESTIONS, start=1):
    run = run_agent(question)
    results.append(run)
    print(f"=== Вопрос #{number} ===")
    if not run["trace"]:
        print(f"Пользователь: {question}")
    for step in run["trace"]:
        print(format_step(step))
    if not any(step["type"] == "answer" for step in run["trace"]):
        print(f"Ответ: {run['answer']}")
    print()

=== Вопрос #1 ===
Пользователь: Какая погода в Тбилиси в мае?
Вызов: get_climate(city='Тбилиси', month=5)
   Результат: В Тбилиси в мае средняя температура 17 °C
Ответ: В Тбилиси в мае средняя температура составляет около 17 °C.

=== Вопрос #2 ===
Пользователь: Какая валюта в Ереване и сколько в ней будет 100 евро?
Вызов: get_city_info(city='Yerevan')
   Результат: Ереван, Армения. Валюта: AMD. Часовой пояс: UTC+4. / Отель: около 25000 AMD за ночь. / Розовый город с видом на Арарат. / Достопримечательности: Каскад (бес
Вызов: convert_currency(amount=100, from_currency='EUR', to_currency='AMD')
   Результат: 100.00 EUR = 42000.00 AMD
Ответ: В Ереване официальная валюта — армянский драм (AMD).  
100 EUR ≈ 42 000 AMD.

=== Вопрос #3 ===
Пользователь: Как добраться из Казани в Белград? Нужен самый дешёвый вариант.
Вызов: search_routes(destination='Belgrade', origin='Kazan', transport='any')
   Результат: Прямых маршрутов не найдено. Попробуйте пересадку через другой город.
Вызов: search_ro

Сводка по прогону. Для каждого вопроса: сколько всего было вызовов инструментов, сколько из них повторов (вызов с теми же аргументами, что уже был), сколько закончились ошибкой. Под строкой — уникальные вызовы каждого инструмента, коротко: только значения аргументов.

In [26]:
from collections import Counter


def short_args(args: dict) -> str:
    """Значения аргументов через «/»: {"origin": "Казань", "destination": "Стамбул"} → «Казань/Стамбул»."""
    return "/".join(str(value) for value in args.values()) or "—"


print(f"{'№':<4}{'вызовов':<9}{'повторов':<10}{'ошибок':<8}инструменты")
for number, run in enumerate(results, start=1):
    calls = [step for step in run["trace"] if step["type"] == "tool_call"]
    keys = [(call["name"], json.dumps(call["args"], ensure_ascii=False, sort_keys=True)) for call in calls]
    repeats = len(keys) - len(set(keys))
    errors = sum(1 for step in run["trace"] if step["type"] == "tool_result" and step["is_error"])
    grouped = ", ".join(
        f"{name} ×{count}" if count > 1 else name for name, count in Counter(call["name"] for call in calls).items()
    )
    print(f"{number:<4}{len(calls):<9}{repeats:<10}{errors:<8}{grouped or '—'}")

    unique: dict[str, list[str]] = {}
    for call in calls:
        values = unique.setdefault(call["name"], [])
        if short_args(call["args"]) not in values:
            values.append(short_args(call["args"]))
    for name, values in unique.items():
        print(f"      {name}: {'; '.join(values)}")

№   вызовов  повторов  ошибок  инструменты
1   1        0         0       get_climate
      get_climate: Тбилиси/5
2   2        0         0       get_city_info, convert_currency
      get_city_info: Yerevan
      convert_currency: 100/EUR/AMD
3   6        1         0       search_routes ×6
      search_routes: Belgrade/Kazan/any; Moscow/Kazan/any; Belgrade/Moscow/any; Istanbul/Moscow/any; Belgrade/Istanbul/any
4   3        0         0       manage_itinerary ×3
      manage_itinerary: add/Istanbul/3; add/Belgrade/2; show
5   3        0         0       search_routes ×3
      search_routes: Dubai/Moscow/train; Istanbul/Moscow/train; Dubai/Moscow/plane
6   0        0         0       —
7   0        0         0       —
8   0        0         0       —


### Выводы по прогону

Посмотрите трейсы и сводку выше и ответьте, ссылаясь на номера вопросов:

1. Какие вопросы агент решил правильно, а где ошибся? Ошибка была в выборе инструмента, в аргументах или в финальном ответе?
2. Справился ли агент с пересадкой (#3)? Если нет — что бы вы поменяли: описание `search_routes`, системный промпт, сам инструмент?
3. Были ли лишние или повторные вызовы инструментов (колонка «повторов» и число вызовов в сводке)? С чем это связано: с промптом, с описаниями инструментов или с тем, что агенту не хватает какого-то инструмента?
4. Что вы меняли в описаниях инструментов или промпте по ходу работы и как это повлияло на поведение?

**Ваш ответ:**

1. Агент правильно решил вопросы 1, 2, 4 и 6-8. В 3 ошибся с количеством пересадок, в 5 сделал лишний поиск самолёта.

2. С 3 агент справился частично: нашел дешевый маршрут, но неправильно посчитал пересадки. Возможно, стоит улучшить search_routes, добавив поиск составных маршрутов.

3. Повторы были в 3, лишний вызов — в 5. Это связано с ограничениями search_routes и недостаточно точными инструкциями.

4. Правила написания валют (все в евро) и обновление системного промпта с ограничениями на инструменты для уменьшения галлюцинаций.

## Перед сдачей

* [ ] Все ячейки выполнены по порядку, выводы видны (`Kernel → Restart & Run All`).
* [ ] Тексты в 2.6, 4.1, 4.3 и выводы по части 5 заполнены.
* [ ] В части 5 напечатаны трейсы по всем тестовым вопросам и сводка.